_У нас есть датасет с большим кол-вом пропусков._

In [1]:
import pandas as pd

data = pd.read_csv('8 ks_crashed.csv')
data.head()

,Название,Главная категория,Валюта,Собрано в долларах,Цель в долларах
0,"Don't Call it a Comeback ""Telescopes""",Music,USD,600.00,600.0
1,Me & You Coordinating Sunglasses- Optical Qual...,NaN,USD,502.00,10000.0
2,New Carts for Istanbul Street Food Vendors,Food,USD,2414.00,1400.0
3,New Improv Comedy Venue in Des Moines,Theater,USD,10030.88,10000.0
4,The Seer and the Sword,Film & Video,USD,0.00,10000.0


_Что будет если выбросить все строки с хотя бы одним пропущенным значением._

In [2]:
data_not_nan = data.dropna(axis=0)

#сколько объектов потеряем?
loss_object = data.shape[0] - data_not_nan.shape[0]
print(f'Потеряем {loss_object} объектов')

#сколько это в процентах?
print(f'Это {round(100 * loss_object / data.shape[0], 1)} % процентов')


Потеряем 89785 объектов
Это 27.1 % процентов


_Узнаем сколько пропусков где находится._

In [18]:
#.sum() складывает значения по одной из осей
data.isna().sum()

Название              3
Главная категория     0
Валюта                0
Собрано в долларах    0
Цель в долларах       0
dtype: int64

### **_Заполним все пропуски в главной категории самой популярной категорией._**
 _Сначала узнаем её._

In [4]:
#узнаем самую популярную категорию
#.value_counts() возвращает серию отсортированную по самому популярному значению
sort_categories = data['Главная категория'].value_counts()

popular_category = sort_categories.index[0]

_Теперь заполним значения._

In [5]:
data['Главная категория'] = data['Главная категория'].fillna(popular_category)

data['Главная категория'].isna().sum()

np.int64(0)

### **_Заполним все пропуски в валюте неизвестной валютой._**

In [8]:
data['Валюта'] = data['Валюта'].fillna('Неизвестная валюта')
#посмотрим на результат
data['Валюта'].isna().sum()

np.int64(0)

### **_Заполним пропуски в единственном вещественном признаке - цель в долларах._**

In [10]:
#можем просто заполнить их нулём
data['Цель в долларах'].fillna(0)

0           600.00
1         10000.00
2          1400.00
3         10000.00
4         10000.00
            ...   
331670     9000.00
331671     2501.45
331672     1500.00
331673        0.00
331674     4000.00
Name: Цель в долларах, Length: 331675, dtype: float64

_Также можем использовать среднее или медиану._

In [17]:
data['Цель в долларах'].fillna(data['Цель в долларах'].mean())
#медианное значение — это значение, которое находится посередине упорядоченного ряда данных.
data['Цель в долларах'].fillna(data['Цель в долларах'].median())

0           600.000000
1         10000.000000
2          1400.000000
3         10000.000000
4         10000.000000
              ...     
331670     9000.000000
331671     2501.450000
331672     1500.000000
331673    98961.220521
331674     4000.000000
Name: Цель в долларах, Length: 331675, dtype: float64

### **_Заполним пропуски в 'Цель в долларах', смотря на похожие объекты._**

_Будем считать объекты похожими, если они относятся к одной категории. Возьмем все объекты той же категории и узнаем, сколько в среднем у них цель в долларах._

In [13]:
#сгруппируем по главной категории и возьмём только цель в долларах
data_cat_group = data.groupby('Главная категория')['Цель в долларах']

#узнаем среднее для каждой категории
#мы не используем .mean(), чтобы сохранить исходный размер таблицы
data_cat_mean = data_cat_group.transform('mean')
#теперь для каждого объекта мы знаем среднее значение его категории
data_cat_mean

0         14706.137908
1         63850.235173
2         45638.010981
3         23878.837883
4         63850.235173
              ...     
331670    18769.529918
331671    14706.137908
331672    14706.137908
331673    98961.220521
331674     8046.212217
Name: Цель в долларах, Length: 331675, dtype: float64

_Заполним._

In [19]:
#если в .fillna() передать series он проведет заполнения в соответствие с индексами
data['Цель в долларах'] = data['Цель в долларах'].fillna(data_cat_mean)


_Проверим общий результат._

In [20]:
data.isna().sum()

Название              3
Главная категория     0
Валюта                0
Собрано в долларах    0
Цель в долларах       0
dtype: int64

_Заберем восстановленые данные._

In [21]:
#index=False — не сохранять индекс DataFrame как отдельный столбец
data.to_csv('ks_восстановленные.csv', index=False)